In [99]:
import os
import json
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS, GPSTAGS

def convert_to_degrees(value):
    """Convert GPS coordinates to degrees in float format"""
    d, m, s = value
    return round(d + (m / 60.0) + (s / 3600.0), 7)

def get_metadata(img):
    """Get all metadata including GPS coordinates if available"""

    exif_data = img.getexif()
    
    metadata = {}
    
    # Get basic EXIF data
    for tag_id, value in exif_data.items():
        tag = TAGS.get(tag_id, tag_id)
        metadata[tag] = value
        
    if metadata.get("Orientation") is None:
        metadata["Orientation"] = 1
    
    # Get GPS data if it exists
    if 34853 in exif_data:  # GPSInfo tag exists
        gps_ifd = exif_data.get_ifd(0x8825)
        gps_info = {}
        for tag_id, value in gps_ifd.items():
            tag = GPSTAGS.get(tag_id, tag_id)
            gps_info[tag] = value
        
        # Convert to decimal degrees if coordinates exist
        if 'GPSLatitude' in gps_info and 'GPSLongitude' in gps_info:
            lat = convert_to_degrees(gps_info['GPSLatitude'])
            lon = convert_to_degrees(gps_info['GPSLongitude'])
            
            if gps_info.get('GPSLatitudeRef') == 'S':
                lat = -lat
            if gps_info.get('GPSLongitudeRef') == 'W':
                lon = -lon
            
            metadata['Latitude'] = lat
            metadata['Longitude'] = lon
            metadata['GPS_raw'] = gps_info
    
    return metadata

def correct_image_orientation(img, orientation):
    """
    Rotate image based on EXIF orientation tag.
    """
    
    if orientation is None:
        return img
    
    # Apply rotation based on orientation value
    if orientation == 3:
        img = img.rotate(180, expand=True)
    elif orientation == 6:
        img = img.rotate(270, expand=True)
    elif orientation == 8:
        img = img.rotate(90, expand=True)
    
    return img

def resize_image_to_target_size(input_path, output_path, target_size_mb=1.0, quality=85):
    """
    Resize an image to approximately target file size.
    
    Args:
        input_path: Path to input image
        output_path: Path to save resized image
        target_size_mb: Target file size in MB (default 1.0)
        quality: Initial JPEG quality (default 85)
    """
    target_size_bytes = target_size_mb * 1024 * 1024
    
    with Image.open(input_path) as img:
        metadata = get_metadata(img)
        
        img = correct_image_orientation(img=img, orientation=metadata.get("Orientation"))
        
        # Convert RGBA to RGB if necessary (for PNG with transparency)
        if img.mode in ('RGBA', 'LA', 'P'):
            background = Image.new('RGB', img.size, (255, 255, 255))
            if img.mode == 'P':
                img = img.convert('RGBA')
            background.paste(img, mask=img.split()[-1] if img.mode in ('RGBA', 'LA') else None)
            img = background
        
        # Start with original dimensions
        width, height = img.size
        
        # Save with initial quality and check size
        img.save(output_path, 'JPEG', quality=quality, optimize=True)
        current_size = os.path.getsize(output_path)
        
        # If still too large, reduce dimensions
        while current_size > target_size_bytes and quality > 10:
            if quality > 50:
                quality -= 5
            else:
                # Reduce dimensions by 10%
                width = int(width * 0.9)
                height = int(height * 0.9)
                img_resized = img.resize((width, height), Image.Resampling.LANCZOS)
                img_resized.save(output_path, 'JPEG', quality=quality, optimize=True)
                current_size = os.path.getsize(output_path)
                continue
            
            img.save(output_path, 'JPEG', quality=quality, optimize=True)
            current_size = os.path.getsize(output_path)
            
        metadata["AspectRatio"] = width / height
        
    return current_size, metadata

def process_images(input_dir, output_dir, target_size_mb=1.0):
    """
    Process all images in input directory and save to output directory.
    """
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)
    
    image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
    
    metadata_records = []
    
    for image_file in input_path.iterdir():
        if image_file.suffix in image_extensions:
            output_file = output_path / f"{image_file.stem}.jpg"
            
            file_name = image_file.name.split(".")[0]
                    
            try:
                original_size = os.path.getsize(image_file)
                final_size, metadata = resize_image_to_target_size(
                    image_file, 
                    output_file, 
                    target_size_mb
                )
                
                fn_split = file_name.split("-") 
                metadata_small = {
                    "FileName": output_file.name,
                    "Chapter": fn_split[0],
                    "Id": fn_split[1],
                    "Description": fn_split[2],
                    "FileSize": final_size,
                    "OriginalSize": original_size
                }
                metadata_small.update(
                    {_k: metadata.get(_k) for _k in ("AspectRatio", "Latitude", "Longitude", "DateTime")}
                )
                
                metadata_records.append(metadata_small)
                
                # print(f"Processed: {image_file.name}")
                # print(f"  Original: {original_size / (1024*1024):.2f} MB")
                # print(f"  Final: {final_size / (1024*1024):.2f} MB")
                # print()
                
            except Exception as e:
                print(f"Error processing {image_file.name}: {e}")
                
    with open(os.path.join(output_dir, "metadata.json"), "w") as _f:
        json.dump(metadata_records, _f, indent=2)

In [100]:
output_folder = os.path.join(os.getcwd(), "src", "assets", "map_data")
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    
raw_photos_path = os.path.join(os.getcwd(), "map-photos-raw")

for _d in Path(raw_photos_path).iterdir():
    if _d.name.startswith("."):
        continue
    path_out = os.path.join(output_folder, _d.name)
    process_images(_d.absolute().__str__(), path_out)